##nc file reading


In [1]:
##prep to view netcdf
import xarray as xr
import matplotlib.pyplot as plt


In [ ]:
pr_file = "../../data/Global_weather/PNWNAmet_pr.nc.nc"
tmin_file = "../../data/Global_weather/PNWNAmet_tasmin.nc.nc"
tmax_file = "../../data/Global_weather/PNWNAmet_tasmax.nc.nc"
dp = xr.open_dataset(pr_file)
dtmin = xr.open_dataset(tmin_file)
dtmax = xr.open_dataset(tmax_file)



<xarray.Dataset> Size: 59MB
Dimensions:  (time: 24837, lat: 19, lon: 31)
Coordinates:
  * time     (time) datetime64[ns] 199kB 1945-01-01 1945-01-02 ... 2012-12-31
  * lat      (lat) float64 152B 49.66 49.59 49.53 49.47 ... 48.66 48.59 48.53
  * lon      (lon) float64 248B -125.1 -125.0 -125.0 ... -123.3 -123.3 -123.2
Data variables:
    tasmax   (time, lat, lon) float32 59MB ...
Attributes:
    history:      Wed Oct 04 13:15:09 2017: cdo -select,name=tasmax pr+tasmax...
    _nc3_strict:  1
    date:         2015-04-05 16:57:30
    notes:        Tps interpolation to 0.0625-deg grid over western North Ame...
    CDO:          Climate Data Operators version 1.6.9 (http://mpimet.mpg.de/...
    CDI:          Climate Data Interface version 1.6.9 (http://mpimet.mpg.de/...
    Conventions:  CF-1.4

In [8]:
##combine the temp to find tavg from 1960-2012
tavg = (dtmin.tasmin + dtmax.tasmax) / 2
##make new dataset with tavg and pr
ds = xr.Dataset({'tavg': tavg, 'pr': dp.pr})
##show extent of dataset
ds

<xarray.Dataset> Size: 91MB
Dimensions:  (lat: 19, lon: 31, time: 19359)
Coordinates:
  * lat      (lat) float64 152B 49.66 49.59 49.53 49.47 ... 48.66 48.59 48.53
  * lon      (lon) float64 248B -125.1 -125.0 -125.0 ... -123.3 -123.3 -123.2
  * time     (time) datetime64[ns] 155kB 1960-01-01 1960-01-02 ... 2012-12-31
Data variables:
    tavg     (time, lat, lon) float32 46MB -2.189 -1.977 -1.592 ... 2.441 2.416
    pr       (time, lat, lon) float32 46MB ...

## Backfilling with Daymet (2013 - 2025)

PNWNAmet stops at **2012-12-31**, so the record is extended with
[Daymet v4 R1](https://doi.org/10.3334/ORNLDAAC/2129) (1 km, North America), which
currently runs through **2025-12-31**.

`pydaymet` / the old `thredds.daac.ornl.gov` NetCDF-subset service now redirect to
**NASA Earthdata Login**, which is where the authentication error comes from. The fix is
an Earthdata account plus a `~/.netrc` entry - not an SSL workaround:

```
machine urs.earthdata.nasa.gov
    login YOUR_USERNAME
    password YOUR_PASSWORD
```
then `chmod 600 ~/.netrc`. Register free at <https://urs.earthdata.nasa.gov/users/new>.

`build_merged_weather.py` pulls Daymet through Earthdata **OPeNDAP** (DAP4 spatial
subsetting, so only our little window is transferred), area-averages the 1 km pixels onto
the PNWNAmet 0.0625 deg grid, and splices the two records together.

Run it once from a terminal:

```bash
python build_merged_weather.py
```


In [ ]:
##load the merged 1960-2025 record built by build_merged_weather.py
merged_file = "../../data/Global_weather/merged_tavg_prcp_1960_2025.nc"
ds_full = xr.open_dataset(merged_file)
ds_full

### Filling the ocean cells (2013 - 2025)

PNWNAmet interpolates over water, so the marine cells are already populated for
**1960 - 2012**. Daymet is land-only, so those same cells go `NaN` from **2013** - that
gap, and only that gap, is what needs filling.

`fill_ocean_cells.py` patches them with **NARR** (NCEP North American Regional Reanalysis,
32 km, 1979 - 2026, served by NOAA PSL with no login). Because NARR overlaps PNWNAmet for
34 years (1979 - 2012), it is bias-corrected against PNWNAmet per cell and per calendar
month - additive offset for `tavg`, multiplicative ratio for `pr` - so the ocean series
stays continuous across 2013 instead of stepping.

```bash
python fill_ocean_cells.py     # run AFTER build_merged_weather.py
```

The `ocean_filled` flag marks which cells were patched.

In [ ]:
##reload after the ocean fill and check what got patched
ds_full = xr.open_dataset(merged_file)

if 'ocean_filled' in ds_full:
    n = int(ds_full.ocean_filled.sum())
    print(f"ocean-filled cells: {n} of {ds_full.ocean_filled.size}")
    print(f"remaining NaN in tavg: {int(ds_full.tavg.isnull().sum())}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ds_full.ocean_filled.plot(ax=ax, cmap='Blues', cbar_kwargs={'label': 'NARR-filled'})
    ax.set_title('Cells patched with NARR from 2013 onward')
    plt.tight_layout(); plt.show()
else:
    print("not filled yet - run: python fill_ocean_cells.py")

### Does the ocean fill line up?

The bias correction is fitted on 1979 - 2012, so 2013 onward is an honest out-of-sample
test. If the correction is working, the marine-cell series should carry across the 2013
boundary without a visible step.

In [ ]:
##marine vs land cells across the 2013 splice
if 'ocean_filled' in ds_full:
    ocean = ds_full.ocean_filled == 1
    ann = ds_full.tavg.groupby('time.year').mean()

    fig, ax = plt.subplots(figsize=(11, 4))
    for mask, lbl, c in [(ocean, 'marine cells (PNWNAmet -> NARR)', 'tab:blue'),
                         (~ocean, 'land cells (PNWNAmet -> Daymet)', 'tab:green')]:
        ax.plot(ann.year, ann.where(mask).mean(dim=['lat', 'lon']), label=lbl, color=c)
    ax.axvline(2012.5, ls='--', c='k', lw=1, label='splice')
    ax.set_xlabel('year'); ax.set_ylabel('mean tavg (°C)')
    ax.set_title('Annual mean temperature: marine vs land cells')
    ax.legend(); plt.tight_layout(); plt.show()

### Caveats

1. **The land splice has a measured +0.85 °C step at 2013.** PNWNAmet and Daymet are
   independent products, and the *land* splice is **not** bias-corrected - only the ocean
   fill is. Domain-mean land temperature jumps +0.85 °C from 2012 to 2013, which is
   comparable to decades of real warming. **Do not read trends across 2013 without
   accounting for this.** The `data_source` coordinate (`0` = PNWNAmet, `1` = Daymet)
   flags which product each day came from.

   For contrast, the bias-corrected marine cells step by only **-0.03 °C** across the same
   boundary, and their 2003-2012 / 2013-2022 means are identical (10.27 °C both). The same
   correction could be applied to land using the 1980-2012 Daymet/PNWNAmet overlap.

2. **5 cells are NaN for the entire record** (48.53-48.59 °N, ~-125.0 °W, southwest
   corner). They fall outside the PNWNAmet domain, so there is no reference to calibrate
   a fill against.

3. **NARR is coarse.** At 32 km it resolves the Strait of Georgia only broadly, so filled
   marine cells are a regional estimate, not a local one.

4. **`tavg` is defined slightly differently in the fill.** PNWNAmet and Daymet use
   `(tmin+tmax)/2`; NARR's daily `air.2m` is a true daily mean. The monthly offset absorbs
   the systematic part of that difference, but they are not identical.

5. **365-day calendar.** Daymet drops 31 December in leap years, so 2016-12-31,
   2020-12-31 and 2024-12-31 are absent (24,104 days, not 24,107).

6. **Upgrade path.** For finer marine resolution, ERA5 (0.25°) via the Copernicus CDS
   beats NARR - it needs a free CDS account and `~/.cdsapirc`.

In [ ]:
##mean precipitation before and after the splice (should now be gap-free)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (sl, lbl) in zip(axes, [(slice('1960', '2012'), 'PNWNAmet 1960-2012'),
                                (slice('2013', '2025'), 'Daymet + NARR 2013-2025')]):
    m = ds_full.pr.sel(time=sl).mean('time')
    im = m.plot(ax=ax, add_colorbar=False, vmin=0, vmax=12, cmap='viridis')
    ax.set_title(lbl)
fig.colorbar(im, ax=axes, label='mean precipitation (mm/day)')
plt.show()